# M1 Notebook 04 — Matrix Algebra and Linear Systems

**Notebook ID:** M1_N04  
**Status:** Runnable first edition  
**Random seed:** 42


## Learning objectives

1. Interpret matrix dimensions and entries.
2. Perform matrix operations.
3. Compute trace and determinants.
4. Solve $A\mathbf{x}=\mathbf{b}$.
5. Implement Gaussian elimination with pivoting.
6. Analyze residuals, rank, and conditioning.
7. Connect matrices to Statistics, AI, and Decision Intelligence.


In [ ]:
# Environment bootstrap — run this cell before the imports below.
# Local VS Code: uses the installed editable srai_math package.
# Google Colab: if srai_math is absent, upload the complete reviewer packet ZIP.
import importlib
import importlib.util
import subprocess
import sys
from pathlib import Path

if importlib.util.find_spec("srai_math") is None:
    try:
        from google.colab import files
    except ImportError as exc:
        raise ModuleNotFoundError(
            'srai_math is not installed. From the extracted packet root, run: '
            'py -m pip install -e ".[dev]"'
        ) from exc

    import zipfile

    print("Upload SRAI_PU-B01-C04_Independent_Reviewer_Packet_v0.3.1.zip")
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
    if len(zip_names) != 1:
        raise RuntimeError("Upload exactly one reviewer packet ZIP.")

    extraction_root = Path("/content/srai_lesson4_reviewer_v013")
    extraction_root.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_names[0]) as archive:
        archive.extractall(extraction_root)

    projects = list(extraction_root.rglob("pyproject.toml"))
    if len(projects) != 1:
        raise RuntimeError("Expected exactly one pyproject.toml in the reviewer packet.")

    project_root = projects[0].parent
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-e", f"{project_root}[dev]"]
    )

    # Editable-install path files are normally processed only when Python starts.
    # Activate the supplied package in the current Colab kernel immediately.
    source_root = str(project_root / "src")
    if source_root not in sys.path:
        sys.path.insert(0, source_root)
    importlib.invalidate_caches()

# READY is printed only after a real import succeeds in the active kernel.
import srai_math
from srai_math.utils import environment_info, set_seed
from srai_math.algebra import gaussian_elimination, residual
print(f"srai_math environment: READY ({Path(srai_math.__file__).resolve()})")


In [ ]:
from srai_math.utils import environment_info, set_seed
from srai_math.algebra import (
    condition_number, determinant_2x2, gaussian_elimination, matrix_add,
    matrix_multiply, rank, residual, trace, transpose,
)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
set_seed(42)
environment_info()


## Matrix foundations

$$
A=[a_{ij}]\in\mathbb{R}^{m\times n}.
$$


In [ ]:
A=np.array([[1.,2.],[3.,4.]])
B=np.array([[5.,6.],[7.,8.]])
A.shape, A


## Addition, transpose, and multiplication

$$
(A+B)_{ij}=a_{ij}+b_{ij},\qquad
(A^\top)_{ij}=a_{ji},
$$

$$
(AB)_{ij}=\sum_k a_{ik}b_{kj}.
$$


In [ ]:
sum_manual=matrix_add(A,B)
prod_manual=matrix_multiply(A,B)
AT=transpose(A)
assert np.allclose(sum_manual,A+B)
assert np.allclose(prod_manual,A@B)
assert np.allclose(AT,A.T)
sum_manual, prod_manual, AT


## Trace and determinant

$$
\operatorname{tr}(A)=\sum_i a_{ii},
\qquad
\det\begin{bmatrix}a&b\\c&d\end{bmatrix}=ad-bc.
$$


In [ ]:
tr=trace(A)
det=determinant_2x2(A)
assert np.isclose(tr,5)
assert np.isclose(det,-2)
tr, det


## Linear systems

$$
A\mathbf{x}=\mathbf{b}.
$$


In [ ]:
A_sys=np.array([[2.,1.],[1.,3.]])
b_sys=np.array([5.,7.])
x_manual=gaussian_elimination(A_sys,b_sys)
x_numpy=np.linalg.solve(A_sys,b_sys)
assert np.allclose(x_manual,x_numpy)
x_manual


## Residual

$$
\mathbf{r}=\mathbf{b}-A\hat{\mathbf{x}}.
$$

This notebook uses the signed residual convention implemented by srai_math.algebra.residual.


In [ ]:
r=residual(A_sys,x_manual,b_sys)
assert np.linalg.norm(r)<1e-12
signed_example=residual(np.array([[2.,0.],[0.,3.]]),np.array([1.1,1.9]),np.array([2.,6.]))
assert np.allclose(signed_example,np.array([-0.2,0.3]))
r


## Partial pivoting experiment

In [ ]:
A_pivot=np.array([[1e-12,1.],[1.,1.]])
b_pivot=np.array([1.,2.])
x=gaussian_elimination(A_pivot,b_pivot)
assert np.allclose(x,np.linalg.solve(A_pivot,b_pivot))
x


## Rank and conditioning

Rank measures independent linear information. The condition number measures sensitivity.


In [ ]:
full=np.eye(2)
deficient=np.array([[1.,2.],[2.,4.]])
ill=np.array([[1.,1.],[1.,1.000001]])
summary={
    "rank_full":rank(full),
    "rank_deficient":rank(deficient),
    "condition_identity":condition_number(full),
    "condition_ill":condition_number(ill),
}
summary


In [ ]:
assert summary["rank_full"]==2
assert summary["rank_deficient"]==1
assert np.isclose(summary["condition_identity"],1)


## Sensitivity of an ill-conditioned system

In [ ]:
b1=np.array([2.,2.000001])
b2=np.array([2.,2.000002])
x1=np.linalg.solve(ill,b1)
x2=np.linalg.solve(ill,b2)
pd.DataFrame({"solution_1":x1,"solution_2":x2,"change":np.abs(x2-x1)})


## Statistics interpretation

A dataset is a matrix $X\in\mathbb{R}^{n\times p}$. Matrix operations support centering, covariance, regression, PCA, and transformations.


In [ ]:
data=pd.DataFrame({
    "rainfall":[400,500,600,700],
    "fertilizer":[20,25,35,40],
    "yield":[2.8,3.4,4.1,4.7],
})
X=data.to_numpy(float)
centered=X-X.mean(axis=0)
assert np.allclose(centered.mean(axis=0),0)
centered


## AI interpretation

Matrices represent weight layers, embeddings, attention scores, Jacobians, Hessians, adjacency structures, and state transitions.


## Decision Intelligence case — Intersectoral influence matrix

In [ ]:
sectors=["Agriculture","Health","Energy","Transport"]
influence=pd.DataFrame([
    [0.70,0.10,0.20,0.15],
    [0.10,0.80,0.25,0.10],
    [0.25,0.10,0.75,0.30],
    [0.20,0.15,0.35,0.70],
],index=sectors,columns=sectors)
state=np.array([0.8,0.7,0.6,0.65])
next_state=influence.to_numpy()@state
pd.Series(next_state,index=sectors,name="modeled_next_state")


In [ ]:
fig,ax=plt.subplots(figsize=(6,5))
im=ax.imshow(influence.to_numpy())
ax.set_xticks(range(len(sectors)),sectors,rotation=45,ha="right")
ax.set_yticks(range(len(sectors)),sectors)
ax.set_title("Illustrative Intersectoral Influence Matrix")
fig.colorbar(im,ax=ax)
plt.tight_layout()
plt.show()


## Engineering notes

- Dense multiplication is typically $O(mnp)$.
- Dense Gaussian elimination is $O(n^3)$.
- Avoid explicit inverses when a direct solve is available.
- Sparse systems require specialized storage and solvers.
- Small residuals do not prove model validity.


## Exercises

### Level A
Explain why matrix multiplication is not generally commutative.

### Level B
Solve a $3\times3$ system manually.

### Level C
Extend Gaussian elimination to multiple right-hand sides.

### Capstone
Construct and solve a resource-allocation system, then test its conditioning and sensitivity.


## Key insight

Matrices encode structured relationships. Linear systems turn those relationships into solvable models, making matrix algebra fundamental to Statistics, AI, Digital Twins, and Decision Intelligence.
